# 第12章 线性近似方法

## 目录
1. 环境搭建
2. 特征构造
3. 线性TD学习
4. 编程题

---
## 1. 环境搭建

In [ ]:
!pip install numpy -q
import numpy as np
import matplotlib.pyplot as plt
import random
plt.rcParams['font.sans-serif'] = ['SimHei']
print("环境搭建完成!")

---
## 2. 特征构造

In [ ]:
def polynomial_features(s, degree=2):
    """多项式特征"""
    f = [1.0]
    for d in range(1, degree+1):
        f.append(s ** d)
    return np.array(f)

def fourier_features(s, n_features=5):
    """傅里叶特征"""
    return np.array([np.cos(np.pi * i * s) for i in range(1, n_features+1)])

def rbf_features(s, centers, sigma=0.5):
    """径向基函数特征"""
    features = [1.0]
    for c in centers:
        features.append(np.exp(-(s - c) ** 2 / (2 * sigma ** 2)))
    return np.array(features)

print("特征构造: 多项式/傅里叶/径向基")

---
## 3. 线性TD学习

In [ ]:
class LinearTD:
    """线性TD学习"""
    def __init__(self, n_features, alpha=0.1, gamma=0.9):
        self.theta = np.zeros(n_features)
        self.alpha = alpha
        self.gamma = gamma
    
    def predict(self, features):
        return np.dot(features, self.theta)
    
    def update(self, f, r, nf, done):
        target = r if done else r + self.gamma * self.predict(nf)
        self.theta += self.alpha * (target - self.predict(f)) * f

def linear_td_demo():
    td = LinearTD(5)
    for _ in range(1000):
        f = polynomial_features(random.random())
        td.update(f, 1.0, f, False)
    print(f"线性TD参数: {td.theta[:3]}")

linear_td_demo()

---
## 4. 编程题

### 编程题1：实现不同的特征构造方法并验证效果

In [ ]:
class LinearApprox:
    """线性函数近似"""
    def __init__(self, n_features, alpha=0.01):
        self.theta = np.zeros(n_features)
        self.alpha = alpha
    def predict(self, f): return np.dot(f, self.theta)
    def fit(self, X, y, n_epochs=500):
        for _ in range(n_epochs):
            for f, target in zip(X, y):
                pred = self.predict(f)
                self.theta += self.alpha * (target - pred) * f

def test_feature_engineering():
    """测试特征构造方法"""
    np.random.seed(42)
    true_func = lambda x: np.sin(2 * np.pi * x)
    X_train = np.random.rand(100)
    y_train = true_func(X_train) + np.random.randn(100) * 0.1
    X_test = np.linspace(0, 1, 100)
    y_test = true_func(X_test)
    results = {}
    # 多项式
    feat_train = np.array([polynomial_features(x, 5) for x in X_train])
    feat_test = np.array([polynomial_features(x, 5) for x in X_test])
    model = LinearApprox(feat_train.shape[1])
    model.fit(feat_train, y_train)
    pred = [model.predict(f) for f in feat_test]
    results['多项式(5阶)'] = np.mean((pred - y_test) ** 2)
    # 傅里叶
    feat_train = np.array([fourier_features(x, 5) for x in X_train])
    feat_test = np.array([fourier_features(x, 5) for x in X_test])
    model = LinearApprox(feat_train.shape[1])
    model.fit(feat_train, y_train)
    pred = [model.predict(f) for f in feat_test]
    results['傅里叶基(5)'] = np.mean((pred - y_test) ** 2)
    # RBF
    centers = np.linspace(0, 1, 7)
    feat_train = np.array([rbf_features(x, centers) for x in X_train])
    feat_test = np.array([rbf_features(x, centers) for x in X_test])
    model = LinearApprox(feat_train.shape[1])
    model.fit(feat_train, y_train)
    pred = [model.predict(f) for f in feat_test]
    results['RBF基(7)'] = np.mean((pred - y_test) ** 2)
    print("特征构造方法对比:")
    for name, mse in results.items():
        print(f"  {name}: MSE={mse:.4f}")

test_feature_engineering()

---

### 编程题2：实现线性TD学习算法的收敛性分析

In [ ]:
class LinearTD2:
    """线性TD学习"""
    def __init__(self, n_features, alpha, gamma=0.9):
        self.theta = np.zeros(n_features)
        self.alpha = alpha
        self.gamma = gamma
    def predict(self, f): return np.dot(f, self.theta)
    def update(self, f, r, nf, done):
        target = r if done else r + self.gamma * self.predict(nf)
        self.theta += self.alpha * (target - self.predict(f)) * f

class MarkovChain:
    """简单马尔可夫链"""
    def __init__(self, n_states=5):
        self.n_states = n_states
        self.state = n_states // 2
        self.P = np.zeros((n_states, n_states))
        for i in range(n_states):
            if i > 0: self.P[i, i-1] = 0.5
            if i < n_states-1: self.P[i, i+1] = 0.5
    def reset(self): self.state = self.n_states // 2; return self.state
    def step(self): self.state = np.random.choice(self.n_states, p=self.P[self.state]); return self.state
    def get_reward(self, s): return 1.0 if s == self.n_states - 1 else 0.0

def test_td_convergence():
    """测试TD收敛性"""
    np.random.seed(42)
    n_states, n_features, gamma = 5, 3, 0.9
    n_episodes, n_runs = 1000, 30
    true_values = [0.0, 0.1, 0.3, 0.6, 1.0]
    results = {}
    for alpha in [0.05, 0.1, 0.2, 0.4]:
        errors = []
        for _ in range(n_runs):
            td = LinearTD2(n_features, alpha, gamma)
            env = MarkovChain(n_states)
            for ep in range(n_episodes):
                s = env.reset()
                f = np.array([1.0, s/n_states, (s/n_states)**2])
                while True:
                    ns = env.step()
                    r = env.get_reward(ns)
                    done = (ns == n_states - 1)
                    nf = np.array([1.0, ns/n_states, (ns/n_states)**2])
                    td.update(f, r, nf, done)
                    f = nf
                    if done: break
                est = [td.predict(np.array([1.0, s/n_states, (s/n_states)**2])) for s in range(n_states)]
                errors.append(np.mean([abs(e - t) for e, t in zip(est, true_values)]))
        results[alpha] = np.mean(errors)
    print("线性TD收敛性分析:")
    for alpha, err in results.items():
        print(f"  alpha={alpha}: 最终误差={err:.4f}")

test_td_convergence()

---

### 编程题3：实现最小二乘策略迭代(LSPI)算法

In [ ]:
class LSPI:
    """最小二乘策略迭代"""
    def __init__(self, n_states, n_actions, gamma=0.9):
        self.n_states, self.n_actions, self.gamma = n_states, n_actions, gamma
        self.theta = np.zeros((n_states, n_actions))
    def get_q(self, s, a): return self.theta[s, a]
    def get_policy(self, s): return np.argmax(self.theta[s])
    def update(self, s, a, r, ns, done):
        target = r if done else r + self.gamma * np.max(self.theta[ns])
        self.theta[s, a] += 0.1 * (target - self.theta[s, a])
    def fit(self, transitions, n_iters=5):
        for _ in range(n_iters):
            for s, a, r, ns, done in transitions:
                self.update(s, a, r, ns, done)

class GridWorldLSPI:
    """网格世界"""
    def __init__(self, size=4):
        self.size = size
    def reset(self): return 0
    def step(self, s, a):
        row, col = s // self.size, s % self.size
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        dr, dc = moves[a]
        row = max(0, min(self.size-1, row + dr))
        col = max(0, min(self.size-1, col + dc))
        ns = row * self.size + col
        r = 1.0 if (row, col) == (self.size-1, self.size-1) else -0.01
        return ns, r, (row, col) == (self.size-1, self.size-1)

def test_lspi():
    """测试LSPI"""
    size = 4
    n_states, n_actions = size * size, 4
    lspi = LSPI(n_states, n_actions)
    env = GridWorldLSPI(size)
    transitions = []
    for _ in range(5000):
        s = env.reset()
        for _ in range(20):
            a = random.randint(0, n_actions - 1)
            ns, r, done = env.step(s, a)
            transitions.append((s, a, r, ns, done))
            s = ns
            if done: break
    lspi.fit(transitions)
    print(f"LSPI: 策略 {[lspi.get_policy(i) for i in range(8)]}")

test_lspi()

print("\n第12章 线性近似方法 - 编程题完成!")